# Baseline

Before any feature engineering: three untuned LightGBM models on the raw application form — external scores only, internal only, and combined. This is the starting line the engineered models are measured against. Results are saved for the after-FE comparison in 12.

In [4]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import json
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from src.data import RAW
from src.models import oof

INTERIM = Path("../data/interim")
app = pd.read_csv(RAW / "application_train.csv").set_index("SK_ID_CURR")
y = app.pop("TARGET")
for c in app.select_dtypes("object").columns:
    app[c] = app[c].astype("category")
ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
internal = [c for c in app.columns if c not in ext]

def gbm():
    return lgb.LGBMClassifier(n_estimators=800, learning_rate=0.03, num_leaves=31, min_child_samples=50,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=2.0,
                              n_jobs=-1, verbose=-1)
app.shape

(307511, 120)

## Three baselines

In [5]:
RUNS = INTERIM / "runs"; RUNS.mkdir(parents=True, exist_ok=True)
base = {}
for name, cols in [("external only", ext), ("internal only", internal), ("combined", app.columns.tolist())]:
    pred = oof(gbm(), app[cols], y)
    pd.Series(pred, index=app.index, name=name).to_pickle(RUNS / f"baseline_{name.replace(' ', '_')}.pkl")
    base[name] = roc_auc_score(y, pred)
    print(f"{name:16s} {len(cols):4d} feats   AUC {base[name]:.4f}")

external only       3 feats   AUC 0.7202
internal only     117 feats   AUC 0.7025
combined          120 feats   AUC 0.7603


## Save

In [6]:
(INTERIM / "baseline.json").write_text(json.dumps(base, indent=2)); base

{'external only': 0.7201511534449502,
 'internal only': 0.7025098162249477,
 'combined': 0.7602538555495111}